# Cellulose Modeler: Python API tutorial

This notebook shows how to build cellulose chains and crystallites directly in Python with **Cellulose Modeler**. The library generates PDB files from independent experimental crystallographic references and does not depend on Cellulose Builder.

This tutorial covers:

- installation from PyPI;
- construction of an isolated chain;
- construction of an 18-chain fibril with the `2,3,4,4,3,2` profile;
- selection of the Iα, Iβ, II, and III_I allomorphs;
- generation of PDB files prepared for CHARMM-GUI;
- surface oxidation with the model inspired by Paajanen *et al.* (2016).

## 1. Installation

The distribution name uses a hyphen, whereas the importable Python module uses an underscore. Run the cell below once in each environment.

In [ ]:
%pip install -U cellulose-modeler

Restart the kernel if requested by your environment. Then import the two functions in the public API.

In [ ]:
from pathlib import Path

from cellulose_modeler import build_structure, write_pdb
import cellulose_modeler

print(f"Cellulose Modeler {cellulose_modeler.__version__}")
OUTPUT_DIR = Path("cellulose_modeler_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

## 2. An isolated chain

`glucose_units` is the degree of polymerization (DP) of each chain. The list `layers=[1]` requests a single chain. Iβ is the default allomorph, but it is declared explicitly here to make the model reproducible.

In [ ]:
single_chain = build_structure(
    glucose_units=20,
    layers=[1],
    allomorph="ibeta",
)

single_path = OUTPUT_DIR / "cellulose_ibeta_single_chain_DP20.pdb"
write_pdb(single_path, single_chain)
print(single_path.resolve())

## 3. An 18-chain, DP 20 Iβ fibril

The transverse profile `2,3,4,4,3,2` contains 18 chains. At DP 20, the structure contains 360 glucose residues.

In [ ]:
layers_18 = [2, 3, 4, 4, 3, 2]

fibril = build_structure(
    glucose_units=20,
    layers=layers_18,
    allomorph="ibeta",
)

fibril_path = OUTPUT_DIR / "cellulose_ibeta_18x20.pdb"
write_pdb(fibril_path, fibril)

n_chains = len(fibril.chains)
n_residues = sum(len(chain.residues) for chain in fibril.chains)
print(f"Chains: {n_chains}; residues: {n_residues}")
print(fibril_path.resolve())

## 4. CHARMM-GUI output profile

For Iα and Iβ, `charmm_gui=True` writes the PDB with the profile validated for recognition of β(1→4) linkages by Glycan Reader. Heavy-atom geometry is preserved; hydroxyl hydrogens are omitted so that they can be reconstructed from the force-field topology.

In [ ]:
charmm_path = OUTPUT_DIR / "cellulose_ibeta_18x20_charmm_gui.pdb"
write_pdb(charmm_path, fibril, charmm_gui=True)
print(charmm_path.resolve())

## 5. Crystalline allomorphs

The accepted identifiers are `ialpha`, `ibeta`, `ii`, and `iii-i`. Iα and Iβ have a validated CHARMM-GUI workflow. II and III_I are generated as independent crystallographic PDB files, but they should not use `charmm_gui=True` because this workflow has not been validated in Glycan Reader.

In [ ]:
for allomorph in ("ialpha", "ibeta", "ii", "iii-i"):
    structure = build_structure(
        glucose_units=20,
        layers=layers_18,
        allomorph=allomorph,
    )
    path = OUTPUT_DIR / f"cellulose_{allomorph}_18x20.pdb"
    write_pdb(path, structure, charmm_gui=allomorph in {"ialpha", "ibeta"})
    print(path)

## 6. TOCNF: 25% of surface C6 sites

Oxidation is currently validated only for Iβ. In the Paajanen model, only one alternating, solvent-facing C6 class is eligible on each surface chain.

For the DP 20 fibril with the `2,3,4,4,3,2` profile:

- there are 12 surface chains;
- there are 240 surface C6 sites in total;
- 120 belong to the eligible alternating class;
- oxidizing 50% of the 120 eligible sites produces 60 carboxylates;
- 60/240 corresponds to 25% of all surface C6 sites.

Therefore, the argument below is `oxidation_degree=0.50`, not 0.25. The random seed records a reproducible spatial realization.

In [ ]:
tocnf = build_structure(
    glucose_units=20,
    layers=layers_18,
    allomorph="ibeta",
    oxidation_degree=0.50,
    oxidation_scope="surface",
    oxidation_model="paajanen",
    oxidation_protonated=False,  # deprotonated COO-
    oxidation_seed=344,
)

oxidized = sum(
    residue.oxidized
    for chain in tocnf.chains
    for residue in chain.residues
)
print(f"Eligible sites: {tocnf.oxidation_eligible_sites}")
print(f"Generated carboxylates: {oxidized}")

In [ ]:
tocnf_path = OUTPUT_DIR / "tocnf_ibeta_paajanen_surface25_seed344_charmm_gui.pdb"
write_pdb(tocnf_path, tocnf, charmm_gui=True)
print(tocnf_path.resolve())

## 7. Protonation state

The argument `oxidation_protonated=False` generates carboxylate (`COO⁻`). To generate neutral carboxylic acid (`COOH`), use `oxidation_protonated=True`. The state should be selected according to the pH, force field, and simulation protocol.

## 8. Reproducibility and citation

Always report the Cellulose Modeler version, allomorph, DP, transverse profile, oxidation model and degree, protonation state, random seed, and whether the CHARMM-GUI profile was used. Also cite the primary crystallographic reference for the selected allomorph and, when applicable, Paajanen *et al.* (2016), DOI: [10.1007/s10570-016-1076-x](https://doi.org/10.1007/s10570-016-1076-x).

Complete data provenance is available in the repository at `cellulose_modeler/data/PROVENANCE.md`.